# Building Shor's Algorithm from Scratch, Part 3: Controlled Modular Multiplication

[Part 1](320_shor_scratch_adder.ipynb) built a plain adder. [Part 2](321_shor_scratch_modulo_adder.ipynb) built modular addition, $a+b \bmod N$. This part builds **controlled modular multiplication**: given a fixed classical constant $a$, compute

$$
\ket{\text{ctrl}}\ket{x}\ket{0} \longrightarrow \ket{\text{ctrl}}\ket{x}\ket{\text{ctrl}\cdot(a x \bmod N)}
$$

i.e. multiply $x$ by the constant $a$, modulo $N$, but only when an extra control qubit is $1$ — leaving the result register untouched (still $\ket 0$) when the control is $0$. This is exactly the operation used, repeatedly, to build **modular exponentiation** next: $a^j \bmod N$ is a sequence of controlled-multiplications by $a, a^2, a^4, a^8, \ldots$, each controlled by one bit of $j$.

Reference: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995).

## The idea: multiplication is repeated conditional addition

Binary long multiplication is: for every bit $x_i$ of $x$, if $x_i=1$, add $a \cdot 2^i$ into the running total. Doing this **modulo $N$** at every step (instead of once at the end) keeps every intermediate value bounded, which matters because our adder registers are only sized to hold values below $N$.

So the circuit is: for $i = 0, \ldots, n-1$, apply Part 2's modulo-adder to add the *constant* $(a \cdot 2^i) \bmod N$ into the result register $b$ — but only on the branches of the superposition where **both** the multiplier's own control qubit *and* $x_i$ are $1$.

That "only when two extra qubits are both 1" requirement means every single gate inside the modulo-adder — including the gates that were already 2- or 3-qubit-controlled in Part 2 — needs *two more* controls stacked on top. We need a general way to add controls to an existing circuit, for an arbitrary number of controls, not just the one-off ancilla trick from Part 2.

In [1]:
!pip install git+https://github.com/blueqat/blueqatSDK

/Users/yuichirominato/.pyenv/shims/pip: line 8: /opt/homebrew/opt/pyenv/bin/pyenv: No such file or directory


## A general multi-controlled-X

The standard technique for an $X$ gate controlled by $k$ qubits ($k \ge 3$) uses $k-2$ scratch ("ancilla") qubits: AND the first two controls into an ancilla with a `CCX`, then AND that ancilla with the next control into a second ancilla, and so on, chaining all $k$ controls down to a single qubit that finally controls the real target. Running the same chain of `CCX` gates backwards afterward restores every ancilla to $\ket 0$ (each `CCX` is its own inverse, so "undo" just means "do it again, in reverse order").

In [2]:
from blueqat import Circuit

def mcx(controls, target, ancillas):
    """X on `target`, controlled by ALL qubits in `controls`. Uses `ancillas` as scratch
    (len(ancillas) >= len(controls) - 2), restored to |0> on exit."""
    circ = Circuit()
    n = len(controls)
    if n == 0:
        circ.x[target]
        return circ
    if n == 1:
        circ.cx[controls[0], target]
        return circ
    if n == 2:
        circ.ccx[controls[0], controls[1], target]
        return circ
    chain = []
    circ.ccx[controls[0], controls[1], ancillas[0]]
    chain.append(ancillas[0])
    for i in range(2, n - 1):
        circ.ccx[chain[-1], controls[i], ancillas[len(chain)]]
        chain.append(ancillas[len(chain)])
    circ.ccx[chain[-1], controls[-1], target]
    for i in range(len(chain) - 1, 0, -1):
        circ.ccx[chain[i - 1], controls[i + 1], chain[i]]
    circ.ccx[controls[0], controls[1], chain[0]]
    return circ

In [3]:
# Sanity check: a 4-controlled-X should fire only when all 4 controls are 1, and leave ancillas clean.
import itertools
controls, target, ancillas = [0, 1, 2, 3], 4, [5, 6]
for bits in itertools.product([0, 1], repeat=4):
    c = Circuit(7)
    for i, bit in enumerate(bits):
        if bit:
            c.x[i]
    c += mcx(controls, target, ancillas)
    state = c.m[:].run(shots=1).most_common(1)[0][0]
    got_target = int(state[7 - 1 - target])
    got_ancillas = [int(state[7 - 1 - a]) for a in ancillas]
    expected = 1 if all(bits) else 0
    status = "OK" if got_target == expected and not any(got_ancillas) else "MISMATCH"
    if status != "OK":
        print(bits, "->", got_target, "expected", expected, "ancillas", got_ancillas, status)
print("all 16 input combinations checked")

all 16 input combinations checked


## Representing circuits as data, not just `Circuit` objects

To add controls to *every gate* of a circuit generically, it's easier to first describe the circuit as a plain list of `(controls, target)` pairs — meaning "flip `target` iff all qubits in `controls` are 1" — and only turn that into an actual `Circuit` (via `mcx`) at the very end. Adding an extra control to the whole circuit then just means appending it to every tuple's `controls` before compiling. This is the same carry/sum/modulo-adder logic as Parts 1-2, just re-expressed this way.

In [4]:
def carry_ops(c, a, b, c_next):
    return [((a, b), c_next), ((a,), b), ((c, b), c_next)]

def carry_inv_ops(c, a, b, c_next):
    return [((c, b), c_next), ((a,), b), ((a, b), c_next)]

def sum_ops(c, a, b):
    return [((a,), b), ((c,), b)]

def add_ops(c, a, b, n):
    """b := a + b (carry register c, b has n+1 qubits). Same construction as Part 1."""
    ops = []
    for i in range(n - 1):
        ops += carry_ops(c[i], a[i], b[i], c[i + 1])
    ops += carry_ops(c[n - 1], a[n - 1], b[n - 1], b[n])
    ops.append(((a[n - 1],), b[n - 1]))
    ops += sum_ops(c[n - 1], a[n - 1], b[n - 1])
    for i in range(n - 2, -1, -1):
        ops += carry_inv_ops(c[i], a[i], b[i], c[i + 1])
        ops += sum_ops(c[i], a[i], b[i])
    return ops

def sub_ops(c, a, b, n):
    """b := b - a: add_ops's gate list, reversed."""
    return list(reversed(add_ops(c, a, b, n)))

def modulo_add_ops(c, a, b, N, t, n):
    """b := (a + b) mod N, as Part 2 -- same five steps, expressed as (controls, target) pairs.
    Native controls only; extra controls (for making the whole thing itself controlled) are added by compile_ops."""
    ops = []
    ops += add_ops(c, a, b, n)                                          # step 1: b += a
    ops += sub_ops(c, N, b, n)                                          # step 2: b -= N
    ops.append(((b[n],), t))                                            # step 3: copy overflow into t
    ops += [(ctrl + (t,), tgt) for ctrl, tgt in add_ops(c, N, b, n)]     # step 4: if t: b += N
    ops += sub_ops(c, a, b, n)                                          # step 5: cleanup...
    ops.append(((), b[n]))
    ops.append(((b[n],), t))
    ops.append(((), b[n]))
    ops += add_ops(c, a, b, n)
    return ops

def compile_ops(ops, extra_controls, ancillas):
    """Turn a (controls, target) list into an actual Circuit, with `extra_controls` appended to every gate."""
    circ = Circuit()
    for controls, target in ops:
        circ += mcx(list(controls) + list(extra_controls), target, ancillas)
    return circ

A quick regression check confirms this refactor still reproduces Part 2's plain (uncontrolled) modulo-adder exactly.

In [5]:
def run_modulo_adder(val_a, val_b, val_N, n, n_ancilla=3):
    c = list(range(n))
    a = list(range(n, 2 * n))
    b = list(range(2 * n, 3 * n + 1))
    N = list(range(3 * n + 1, 4 * n + 1))
    t = 4 * n + 1
    ancillas = list(range(4 * n + 2, 4 * n + 2 + n_ancilla))
    circ = Circuit(4 * n + 2 + n_ancilla)
    for i in range(n):
        if (val_a >> i) & 1: circ.x[a[i]]
        if (val_b >> i) & 1: circ.x[b[i]]
        if (val_N >> i) & 1: circ.x[N[i]]
    circ += compile_ops(modulo_add_ops(c, a, b, N, t, n), [], ancillas)
    total = circ.n_qubits
    state = circ.m[:].run(shots=1).most_common(1)[0][0]
    bits = [state[total - 1 - q] for q in b]
    return int("".join(reversed(bits)), 2)

for x, y, N in [(7, 6, 11), (14, 13, 15), (3, 5, 9)]:
    result = run_modulo_adder(x, y, N, 4)
    print(f"({x}+{y}) mod {N} = {result}  (expected {(x + y) % N}, {'OK' if result == (x + y) % N else 'MISMATCH'})")

(7+6) mod 11 = 2  (expected 2, OK)


(14+13) mod 15 = 12  (expected 12, OK)


(3+5) mod 9 = 8  (expected 8, OK)


## The controlled multiplier

For each bit $i$ of $x$: load the classical constant $(a \cdot 2^i) \bmod N$ into a small scratch register with plain `X` gates (this is circuit-construction-time information — $a$, $N$ and $i$ are all known constants — not something that depends on any superposition), run `modulo_add_ops` controlled by *both* the multiplier's control qubit and $x_i$, then undo the `X` gates to reset the scratch register to $\ket 0$ for the next bit.

In [6]:
def build_controlled_multiplier(mult_a, val_x, val_N, ctrl_value, n, n_ancilla=4):
    c = list(range(n))                                # carry register
    const = list(range(n, 2 * n))                      # scratch: holds (a * 2**i) mod N
    b = list(range(2 * n, 3 * n + 1))                  # result register (n+1 wide)
    N = list(range(3 * n + 1, 4 * n + 1))               # modulus
    x = list(range(4 * n + 1, 5 * n + 1))               # multiplicand
    ctrl = 5 * n + 1                                    # multiplier's control qubit
    t = 5 * n + 2                                       # modulo-adder's overflow flag
    ancillas = list(range(5 * n + 3, 5 * n + 3 + n_ancilla))
    circ = Circuit(5 * n + 3 + n_ancilla)

    if ctrl_value:
        circ.x[ctrl]
    for i in range(n):
        if (val_x >> i) & 1:
            circ.x[x[i]]
        if (val_N >> i) & 1:
            circ.x[N[i]]

    for i in range(n):
        const_val = (mult_a * (1 << i)) % val_N
        for j in range(n):
            if (const_val >> j) & 1:
                circ.x[const[j]]
        circ += compile_ops(modulo_add_ops(c, const, b, N, t, n), [ctrl, x[i]], ancillas)
        for j in range(n):
            if (const_val >> j) & 1:
                circ.x[const[j]]

    return circ, b

In [7]:
def run_controlled_multiplier(mult_a, val_x, val_N, ctrl_value, n=None):
    """a, x, N, ctrl can be any values you like -- n (register width) is inferred from N and x
    automatically if not given explicitly."""
    if n is None:
        n = max(val_N, val_x, 1).bit_length()
    circuit, b = build_controlled_multiplier(mult_a, val_x, val_N, ctrl_value, n)
    total = circuit.n_qubits
    print(f"  [n={n}, qubits={total}, gates={len(circuit.ops)}]")
    state = circuit.m[:].run(shots=1).most_common(1)[0][0]
    bits = [state[total - 1 - q] for q in b]
    return int("".join(reversed(bits)), 2)

# n=2 (N up to 3) keeps this well within what this direct simulation approach can handle quickly.
for a, x, N, ctrl in [(2, 1, 3, 1), (2, 2, 3, 1), (2, 0, 3, 1), (2, 1, 3, 0)]:
    result = run_controlled_multiplier(a, x, N, ctrl)
    expected = (a * x) % N if ctrl else 0
    print(f"ctrl={ctrl}: {a}*{x} mod {N} = {result}  (expected {expected}, {'OK' if result == expected else 'MISMATCH'})")

  [n=2, qubits=17, gates=620]


ctrl=1: 2*1 mod 3 = 2  (expected 2, OK)
  [n=2, qubits=17, gates=620]


ctrl=1: 2*2 mod 3 = 1  (expected 1, OK)
  [n=2, qubits=17, gates=619]


ctrl=1: 2*0 mod 3 = 0  (expected 0, OK)
  [n=2, qubits=17, gates=619]


ctrl=0: 2*1 mod 3 = 0  (expected 0, OK)


## Trying a less tiny example

`run_controlled_multiplier` takes any `a`, `x`, `N`, so nothing above is hard-coded to $N=3$ specifically — it just happens to be the size that finishes in a couple of seconds. Since the circuit is built entirely from ordinary Python (no quantum superposition needed to *construct* it), we can print out exactly how many qubits and gates a bigger case would need before running it, and then actually run one modest step up, $N=7$: this takes noticeably longer (about two minutes on the machine this notebook was written on) because gate count grows fast, not because $N=7$ needs more qubits than $N=3$.

In [8]:
import time

t0 = time.time()
result = run_controlled_multiplier(3, 5, 7, 1)  # 3 * 5 mod 7, controlled bit on
print(f"3*5 mod 7 = {result}  (expected {(3 * 5) % 7}, {'OK' if result == (3 * 5) % 7 else 'MISMATCH'})"
      f"  [{time.time() - t0:.0f}s]")

  [n=3, qubits=22, gates=1464]


3*5 mod 7 = 1  (expected 1, OK)  [161s]


All four cases check out, including the important negative case: when `ctrl=0`, the result register stays exactly `0` no matter what `x` is — confirming the whole multiplication is genuinely conditional, not just its final step.

## A word on scale

The $N=7$ example above already took a couple of minutes, for a circuit with only 22 qubits. The reason isn't qubit count — it's **gate count**: the circuit repeats the whole 5-step modulo-adder once per bit of $x$, and every gate inside it picks up two more controls, each of which (for 3+ total controls) costs several extra `CCX` gates via `mcx`. Qubit count grows only *linearly* with $n$, but this simulator's cost per shot grows much faster, since it has to apply every one of those gates in sequence.

There's also a hard wall, not just a slow one: this SDK's shot-sampling backend caps out at $2^{24}$ possible outcomes, and $N=15$ (the classic first example for Shor's algorithm, needing $n=4$) already needs 27 qubits here — $2^{27}$ basis states, past that cap. Getting to $N=15$ would need a smarter simulation strategy (e.g. not expanding the full state vector) or, of course, an actual quantum computer, where the cost of running the circuit doesn't grow with the *simulator's* limitations at all. That gap — easy to build the circuit, increasingly hard to classically simulate what it does as it grows — is a small, hands-on taste of exactly why factoring is believed to be classically hard in the first place.

## What's next

With controlled multiplication in hand, the next piece is **[modular exponentiation](324_shor_scratch_modular_exponentiation.ipynb)** — repeated controlled squaring, using this circuit as the core building block, controlled by each bit of the phase-estimation register in turn. That's the operation Shor's algorithm's quantum part actually runs, and the piece that finally connects this whole series back to [310_shor.ipynb](310_shor.ipynb)'s order-finding theory.